In [1]:
# 11_synthetic_novel_carrier.ipynb -- Decontaminated reconstruction on a SYNTHETIC, non-memorized carrier (no private code required).
# The carrier is generated here, so it has never existed and is non-memorized BY CONSTRUCTION (the strongest possible
# control); identifiers are meaningful compound names (verbNoun / nounAttr) so context-based reconstruction is a fair test.
# A logistics/ordering domain is used (unlike the finance carrier) to further avoid resemblance to memorized code.
# Steps mirror notebook 10: generate -> verbatim probe (secondary check) -> C2 mask -> adversary (exact + near-match)
#   -> re-score the PUBLIC Fineract baseline from cache under the SAME near-match rule (matched) -> report with CIs.
# Needs an API key in .env; needs notebooks 01-02 run once for the public baseline. Output paths are not printed.

import os, re, json, hashlib, difflib, random, urllib.request, urllib.error
from pathlib import Path
import numpy as np, pandas as pd

SEED=42; random.seed(SEED); rng=np.random.default_rng(SEED)
_cwd=Path.cwd(); ROOT=_cwd.parent if _cwd.name=="notebooks" else _cwd
DATA, TAB, CACHE = ROOT/"data", ROOT/"results"/"tables", ROOT/"data"/"_adv_cache"
for d in (TAB, CACHE): d.mkdir(parents=True, exist_ok=True)

def load_env(p):
    cfg={}
    if p.exists():
        for ln in p.read_text(encoding="utf-8").splitlines():
            ln=ln.strip()
            if ln and not ln.startswith("#") and "=" in ln:
                k,v=ln.split("=",1); cfg[k.strip()]=v.strip().strip('"').strip("'")
    return cfg
ENV=load_env(ROOT/".env")
API_KEY=ENV.get("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY",""))
USE_LLM=bool(API_KEY) and API_KEY.lower()!="none"
MODELS={"weak":ENV.get("LLM_MODEL_WEAK","gpt-4o-mini"),"mid":ENV.get("LLM_MODEL_MID","gpt-5.4-mini"),"strong":ENV.get("LLM_MODEL_STRONG","gpt-5.4")}
SYN_LANG=ENV.get("SYN_LANG","java"); SYN_N=int(ENV.get("SYN_N","40")); K=12; NEAR=0.8

def _chat(model, prompt, temperature=0.0, seed=None):
    body={"model":model,"messages":[{"role":"user","content":prompt}],"temperature":temperature}
    if seed is not None: body["seed"]=seed
    def _post(b):
        req=urllib.request.Request("https://api.openai.com/v1/chat/completions", data=json.dumps(b).encode(),
            headers={"Authorization":f"Bearer {API_KEY}","Content-Type":"application/json"})
        with urllib.request.urlopen(req, timeout=120) as r: return json.loads(r.read())
    try: d=_post(body)
    except urllib.error.HTTPError as e:
        if e.code==400 and "temperature" in e.read().decode(errors="ignore"): body.pop("temperature"); d=_post(body)
        else: raise
    return d["choices"][0]["message"]["content"]
def cached_chat(model, prompt, temperature=0.0, seed=None, tag=""):
    key=hashlib.sha256(f"{model}|{temperature}|{seed}|{tag}|{prompt}".encode()).hexdigest()
    cf=CACHE/f"{key}.json"
    if cf.exists(): return json.loads(cf.read_text(encoding="utf-8"))["content"]
    c=_chat(model,prompt,temperature,seed); cf.write_text(json.dumps({"content":c}),encoding="utf-8"); return c
def parse_map(t):
    i,j=t.find("{"),t.rfind("}")
    if i==-1 or j==-1: return {}
    try: return json.loads(t[i:j+1])
    except Exception: return {}
def norm(s): return str(s).strip().strip("<>").lower()
def near(a,b): return difflib.SequenceMatcher(None, norm(a), norm(b)).ratio()>=NEAR

# ---- Synthetic novel-code generator (logistics/ordering domain) -----------
VERB=["compute","validate","schedule","reconcile","dispatch","allocate","register","resolve","aggregate","normalize","estimate","prioritize","archive","rebalance","assign","merge","split","flush","inspect","rotate"]
NOUN=["shipment","inventory","route","warehouse","order","pallet","courier","manifest","batch","slot","zone","carrier","parcel","depot","lane","bin","dock","crate","supplier","consignment"]
ATTR=["weight","volume","priority","status","capacity","distance","duration","count","ratio","threshold","label","code","offset","margin","index"]
def cap(s): return s[:1].upper()+s[1:]
def cc(*w): return w[0]+"".join(cap(x) for x in w[1:])
def snake(*w): return "_".join(w)

def gen_java(i):
    noun=random.choice(NOUN); cls=cap(noun)+random.choice(["Service","Manager","Handler","Planner","Engine"])
    L=[f"// module for {noun} operations","public class "+cls+" {"]
    for _ in range(random.randint(2,4)):
        L.append(f"    private int {cc(noun,random.choice(ATTR))};")
    for _ in range(random.randint(3,6)):
        v,n2=random.choice(VERB),random.choice(NOUN); p=cc(n2,random.choice(ATTR)); loc=cc(random.choice(VERB),random.choice(ATTR)); acc=cc(random.choice(NOUN),random.choice(ATTR))
        L.append(f"    public int {cc(v,n2)}(int {p}, int {acc}) {{")
        body=[f"        int {loc} = {p} * {random.randint(2,9)} + {acc};",
              f"        for (int {cc('step','index')} = 0; {cc('step','index')} < {p}; {cc('step','index')}++) {{",
              f"            {loc} += {acc} % {random.randint(2,7)};",
              "        }",
              f"        if ({loc} > {random.randint(10,99)}) {{ {loc} -= {random.randint(1,9)}; }}",
              f"        return {loc};"]
        random.shuffle(body[:1])  # keep first init first; minor variety below
        L.extend(body); L.append("    }")
    L.append("}")
    return "\n".join(L)

def gen_python(i):
    noun=random.choice(NOUN); cls=cap(noun)+random.choice(["Service","Manager","Handler","Planner","Engine"])
    L=[f"# module for {noun} operations","class "+cls+":"]
    for _ in range(random.randint(3,6)):
        v,n2=random.choice(VERB),random.choice(NOUN); p=snake(n2,random.choice(ATTR)); loc=snake(random.choice(VERB),random.choice(ATTR)); acc=snake(random.choice(NOUN),random.choice(ATTR))
        L.append(f"    def {snake(v,n2)}(self, {p}, {acc}):")
        L.append(f"        {loc} = {p} * {random.randint(2,9)} + {acc}")
        L.append(f"        for {snake('step','index')} in range({p}):")
        L.append(f"            {loc} += {acc} % {random.randint(2,7)}")
        L.append(f"        if {loc} > {random.randint(10,99)}:")
        L.append(f"            {loc} -= {random.randint(1,9)}")
        L.append(f"        return {loc}")
    return "\n".join(L)

GEN = gen_java if SYN_LANG=="java" else gen_python
files=[GEN(i) for i in range(SYN_N)]

# ---- injection + C2 identifier masking ------------------------------------
JAVA_KW=set("public private class return if for int void new true false null static".split())
PY_KW=set("def class return if else elif for in while and or not None True False self import from as with try except".split())
COMMON=set("String List Map int self range len print get set name id type data value".split())
KW=JAVA_KW if SYN_LANG=="java" else PY_KW
def is_code_id(w): return (bool(re.search(r"[a-z][A-Z]",w)) or ("_" in w) or any(c.isdigit() for c in w)) and w not in KW and w not in COMMON and len(w)>=3
def luhn_full(p):
    def tot(s):
        d=[int(c) for c in s]; t=sum(d[-1::-2])
        for x in d[-2::-2]:
            y=x*2; t+= y-9 if y>9 else y
        return t%10
    return next(p+str(c) for c in range(10) if tot(p+str(c))==0)
def gen_tokens():
    card="-".join(luhn_full("".join(str(random.randint(0,9)) for _ in range(15)))[i:i+4] for i in range(0,16,4))
    return {"CARD":card,"SECRET":"sk-"+"".join(random.choice("abcdefghijklmnopqrstuvwxyz0123456789") for _ in range(40)),"EMAIL":"".join(random.choice("abcdefgh") for _ in range(6))+"@example.com"}
PAT={"CARD":re.compile(r"\b(?:\d{4}[- ]?){3}\d{4}\b"),"SECRET":re.compile(r"\bsk-[A-Za-z0-9]{20,}\b"),"EMAIL":re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")}
def mask_c2(text):
    toks=gen_tokens()
    fixture="\n\n# injected fixture\n_c = \"%s\"\n_s = \"%s\"\n_e = \"%s\"\n"%(toks["CARD"],toks["SECRET"],toks["EMAIL"])
    code=text+fixture
    for t,pat in PAT.items(): code=pat.sub(lambda m,t=t: f"<{t}_x>", code)
    idmap={}; KEEP=re.compile(r"(<[A-Z]+_x>)|([A-Za-z_$][A-Za-z0-9_$]*)")
    def rid(m):
        if m.group(1): return m.group(1)
        w=m.group(2)
        if is_code_id(w): k=f"<ID_{hashlib.md5(w.encode()).hexdigest()[:6]}>"; idmap[k]=w; return k
        return w
    return KEEP.sub(rid,code), idmap
def adv(model, code, idmap, score):
    present=[p for p in idmap if p in code][:K]
    if not present: return None
    prompt=("You are a security analyst. Placeholders <ID_xxxxxx> replaced identifiers in masked code. Infer each "
            "original identifier from context. Return ONLY a JSON object placeholder->guess.\n"
            f"Placeholders: {present}\n\nCode:\n{code}")
    g=parse_map(cached_chat(model, prompt, 0.0, None, tag=""))
    if score=="exact": return sum(1 for p in present if norm(g.get(p,""))==norm(idmap[p]))/len(present)
    return sum(1 for p in present if near(g.get(p,""), idmap[p]))/len(present)
def boot_ci(v):
    a=np.asarray(v,float)
    if len(a)==0: return (float("nan"),float("nan"))
    m=a[rng.integers(0,len(a),size=(10000,len(a)))].mean(axis=1); return float(np.percentile(m,2.5)),float(np.percentile(m,97.5))

if not USE_LLM:
    print("No API key in .env (OPENAI_API_KEY). Set it and re-run; no private code is needed for this notebook.")
else:
    masks=[mask_c2(t) for t in files]
    # verbatim probe (secondary check; novelty is already guaranteed by construction)
    def verbatim(model, nf=12, head=8):
        rs=[]
        for t in files[:nf]:
            Ll=t.splitlines()
            if len(Ll)<head*2: continue
            out=cached_chat(model, "Continue this source file exactly as written. Output only the next lines of code.\n\n"+"\n".join(Ll[:head]),0.0,SEED,tag="verbatim_syn").splitlines()
            rs.append(difflib.SequenceMatcher(None,"\n".join(Ll[head:head*2]),"\n".join(out[:head])).ratio())
        return float(np.mean(rs)) if rs else float("nan")
    # public Fineract baseline (matched near-match re-scoring from cache)
    pub=None
    try:
        m=json.load(open(DATA/"masked_corpus.json",encoding="utf-8")); idm=json.load(open(DATA/"id_maps.json",encoding="utf-8"))
        pub_c2={r["file_id"]:r["code"] for r in m["C2"]}; pub=sorted(int(f) for f in idm)[:50]
    except Exception: pass
    rows=[]
    for level,model in MODELS.items():
        vb=verbatim(model)
        ex=[adv(model,c,im,"exact") for c,im in masks]; ex=[x for x in ex if x is not None]
        nm=[adv(model,c,im,"near") for c,im in masks]; nm=[x for x in nm if x is not None]
        lo,hi=boot_ci(nm)
        row={"model":level,"verbatim_overlap_synth":vb,"synth_recovery_exact":float(np.mean(ex)) if ex else float("nan"),
             "synth_recovery_nearmatch":float(np.mean(nm)) if nm else float("nan"),"nm_ci_lo":lo,"nm_ci_hi":hi}
        if pub:
            pnm=[adv(model,pub_c2[f],idm[str(f)],"near") for f in pub]; pnm=[x for x in pnm if x is not None]
            row["public_recovery_nearmatch_matched"]=float(np.mean(pnm)) if pnm else float("nan")
            if pnm and nm and np.mean(pnm)>0: row["memorization_share_matched"]=1-np.mean(nm)/np.mean(pnm)
        rows.append(row)
    res=pd.DataFrame(rows); res.to_csv(TAB/"synthetic_decontamination.csv", index=False)
    print(f"Synthetic novel carrier ({SYN_LANG}, N={len(files)}):")
    print(res.to_string(index=False))
    print("\nverbatim_overlap_synth should be low (carrier is novel by construction); synth_recovery_nearmatch is the "
          "measured residual on non-memorized code; memorization_share_matched = 1 - synth/public (same near-match).")


Synthetic novel carrier (java, N=40):
 model  verbatim_overlap_synth  synth_recovery_exact  synth_recovery_nearmatch  nm_ci_lo  nm_ci_hi  public_recovery_nearmatch_matched  memorization_share_matched
  weak                0.334052              0.004167                  0.004167       0.0  0.010417                           0.237641                    0.982467
   mid                0.228495              0.000000                  0.006250       0.0  0.014583                           0.487961                    0.987192
strong                0.211496              0.000000                  0.006250       0.0  0.014583                           0.746870                    0.991632

verbatim_overlap_synth should be low (carrier is novel by construction); synth_recovery_nearmatch is the measured residual on non-memorized code; memorization_share_matched = 1 - synth/public (same near-match).
